# Notebook 04 — CoreML Export

Exporting YOLOv8n to CoreML for on-device inference on iPhone via Apple Neural Engine.

**Table of Contents**
1. [Export Strategy — Why YOLOv8 not DETR](#part-1)
2. [YOLOv8 CoreML Export](#part-2)
3. [Inspect the Model](#part-3)
4. [Latency Benchmark](#part-4)
5. [Demo Inference](#part-5)

---

<a id='part-1'></a>
## Part 1 — Export Strategy: Why YOLOv8, Not DETR

In app-01 we exported a ViT classification model to CoreML. Detection is harder — the model has two output heads (boxes and classes), and not all detection architectures export cleanly.

### Why YOLOv8 and not DETR for CoreML

| | DETR | YOLOv8n |
|---|---|---|
| Computation graph | Dynamic (Hungarian matcher has control flow) | Static (fixed graph) |
| TorchScript trace | Fails — can't trace dynamic control flow | Passes cleanly |
| Parameters | 167M | 3.2M (~50× smaller) |
| Ultralytics CoreML support | No | Yes — native one-line export |
| On-device suitability | No | Yes |

**DETR's problem:** The Hungarian algorithm (used to match predicted boxes to ground-truth during training) contains dynamic control flow — loops and conditionals that depend on the input. `torch.jit.trace` records a single execution path and replays it. If the control flow changes across inputs, the trace is wrong. You'd need `torch.jit.script` instead, but DETR wasn't designed for that.

**YOLOv8's advantage:** Fixed grid-based predictions, no dynamic matching at inference time. The graph is always the same shape regardless of input. Traces perfectly.

### What changes vs app-01

- **app-01 (classification):** single output tensor — class probabilities, shape `(1, 1000)`
- **app-02 (detection):** two output tensors — bounding boxes + class scores
- Detection also requires **Non-Maximum Suppression (NMS)** to filter overlapping boxes

### Two export options

**Option 1 — NMS built into CoreML model:**
- Model outputs final filtered boxes directly
- Swift code is simple — just read the outputs
- We use this approach

**Option 2 — NMS in Swift:**
- Model outputs raw predictions (e.g. 8400 candidate boxes)
- Swift must implement NMS manually
- More control, more code

We use **Option 1** (`nms=True` in the export call) for simplicity.

In [ ]:
from ultralytics import YOLO
from PIL import Image
import coremltools as ct
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import torch
import requests
from pathlib import Path

assets_dir = Path('../assets')
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)
assets_dir.mkdir(exist_ok=True)

print(f'coremltools: {ct.__version__}')
print(f'ultralytics: installed')

<a id='part-2'></a>
## Part 2 — YOLOv8 CoreML Export

In app-01, we exported manually:
```python
traced = torch.jit.trace(model, example_input)
mlmodel = ct.convert(traced, inputs=[ct.TensorType(...)], ...)
```

Ultralytics wraps all of that for us. One line handles:
1. Loading the model weights
2. Running `torch.jit.trace` with the right dummy input
3. Calling `coremltools.convert` with the right input/output specs
4. Optionally appending an NMS post-processing layer
5. Saving the `.mlpackage`

The result is a `.mlpackage` — Apple's modern CoreML format (introduced in Xcode 13). It's a directory bundle that contains the model weights, compute plan, and metadata. Older CoreML used `.mlmodel` (single file). Always prefer `.mlpackage` for new projects.

In [ ]:
model = YOLO('yolov8n.pt')

# Export to CoreML with NMS built-in
# nms=True  → model outputs filtered boxes directly (no post-processing needed in Swift)
# imgsz=640 → standard YOLO input size
export_path = model.export(format='coreml', nms=True, imgsz=640)
print(f'Exported to: {export_path}')

import shutil
dest = models_dir / 'yolov8n.mlpackage'
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(export_path, dest)
print(f'Copied to: {dest}')

<a id='part-3'></a>
## Part 3 — Inspect the Model

Before deploying to iPhone, always inspect the CoreML model spec. Three things to check:

1. **Input shape** — must match what your Swift `CVPixelBuffer` or `MLMultiArray` will provide. YOLOv8 expects a 640×640 image input.
2. **Output names and shapes** — your Swift code needs to read these by name. With `nms=True`, you'll see `confidence` and `coordinates` (or similar names depending on coremltools version).
3. **Model size** — should be ~6MB for YOLOv8n. Anything over 50MB is a red flag for on-device use.

In [ ]:
cml = ct.models.MLModel(str(models_dir / 'yolov8n.mlpackage'))

spec = cml.get_spec()
print("=== Inputs ===")
for inp in spec.description.input:
    print(f"  {inp.name}: {list(inp.type.imageType.width if inp.type.HasField('imageType') else inp.type.multiArrayType.shape)}")

print("\n=== Outputs ===")
for out in spec.description.output:
    print(f"  {out.name}")

print(f"\n=== Metadata ===")
for k, v in cml.user_defined_metadata.items():
    print(f"  {k}: {v}")

import os
size_mb = sum(f.stat().st_size for f in (models_dir / 'yolov8n.mlpackage').rglob('*') if f.is_file()) / 1e6
print(f"\nModel size: {size_mb:.1f} MB")

<a id='part-4'></a>
## Part 4 — Latency Benchmark

Same benchmarking approach as app-01: warmup runs first (to load the model onto the accelerator), then timed runs.

**Compute unit configs:**
- `ALL` — CoreML picks the best accelerator automatically. On Apple Silicon, this routes to the Neural Engine (ANE). **This is what iPhone uses.**
- `CPU_AND_NE` — CPU + Neural Engine, no GPU. Good for memory-constrained situations.
- `CPU_ONLY` — baseline, slowest, but most predictable.

**Mac vs iPhone:** Mac benchmarks here run on Apple Silicon (M-series). iPhone has the same ANE architecture but a different chip (A-series). Numbers won't be identical, but the relative ordering (ALL < CPU_AND_NE < CPU_ONLY) holds on device.

**Why warmup matters:** The first inference call triggers model compilation and hardware loading. Without warmup, your timing includes one-time setup cost — not representative of steady-state performance.

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_pil = Image.open(requests.get(url, stream=True).raw).resize((640, 640))
img_array = np.array(image_pil)

configs = {
    'ALL': ct.ComputeUnit.ALL,
    'CPU_AND_NE': ct.ComputeUnit.CPU_AND_NE,
    'CPU_ONLY': ct.ComputeUnit.CPU_ONLY,
}

results = {}
for name, compute_unit in configs.items():
    m = ct.models.MLModel(str(models_dir / 'yolov8n.mlpackage'), compute_units=compute_unit)
    
    # Warmup
    for _ in range(5):
        _ = m.predict({'image': image_pil})
    
    # Timed runs
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        _ = m.predict({'image': image_pil})
        times.append((time.perf_counter() - t0) * 1000)
    
    results[name] = {'mean_ms': np.mean(times), 'std_ms': np.std(times)}
    print(f'{name}: {results[name]["mean_ms"]:.1f} ± {results[name]["std_ms"]:.1f} ms')

# PyTorch baseline
yolo_pt = YOLO('yolov8n.pt')
times_pt = []
for _ in range(20):
    t0 = time.perf_counter()
    _ = yolo_pt(image_pil, verbose=False)
    times_pt.append((time.perf_counter() - t0) * 1000)
print(f'PyTorch MPS: {np.mean(times_pt):.1f} ± {np.std(times_pt):.1f} ms')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels = list(results.keys()) + ['PyTorch\nMPS']
means = [r['mean_ms'] for r in results.values()] + [np.mean(times_pt)]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']

bars = ax.bar(labels, means, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}ms', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Mean latency (ms)')
ax.set_title('YOLOv8n CoreML Inference Latency', fontsize=13)
ax.set_ylim(0, max(means) * 1.3)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(assets_dir / 'yolo_coreml_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

<a id='part-5'></a>
## Part 5 — Demo Inference

Run the CoreML model end-to-end and parse the output boxes.

**Output format with `nms=True`:**
- `confidence`: shape `(1, num_detections, 80)` — class scores for each surviving box
- `coordinates`: shape `(1, num_detections, 4)` — box coordinates in `(cx, cy, w, h)` normalized to `[0, 1]`

To get the predicted class for each box: `argmax` over the 80 class scores.

To convert normalized `(cx, cy, w, h)` to pixel `(x1, y1, x2, y2)` for drawing:
```
x1 = (cx - w/2) * image_width
y1 = (cy - h/2) * image_height
x2 = (cx + w/2) * image_width
y2 = (cy + h/2) * image_height
```

Note: the exact output key names (`confidence`, `coordinates`) may vary slightly depending on coremltools version. The code below inspects the keys at runtime and handles both cases.

In [ ]:
# Load model with ALL compute units
cml_model = ct.models.MLModel(str(models_dir / 'yolov8n.mlpackage'), compute_units=ct.ComputeUnit.ALL)

# COCO class names (80 classes)
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
    'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
    'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# Run inference — model takes PIL image input (640x640)
output = cml_model.predict({'image': image_pil})
print("Output keys:", list(output.keys()))

# Parse output — YOLOv8 CoreML NMS output format:
# 'confidence': (1, num_boxes, 80) class scores
# 'coordinates': (1, num_boxes, 4) boxes in (cx, cy, w, h) normalized
for k, v in output.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: {v.shape}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
# Get original image for visualization
orig_image = Image.open(requests.get(url, stream=True).raw)
W, H = orig_image.size

# Parse output — handle both possible output formats
output_arrays = {k: np.array(v) for k, v in output.items()}
keys = list(output_arrays.keys())
print(f"Output keys: {keys}")
print(f"Output shapes: {[(k, output_arrays[k].shape) for k in keys]}")

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(orig_image)
ax.set_title('YOLOv8n CoreML \u2014 On-Device Inference', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(assets_dir / 'yolo_coreml_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print("CoreML model output parsed successfully.")
print(f"Model ready for iPhone deployment.")

## Summary

**What we did:**
- Exported YOLOv8n to CoreML in one line via Ultralytics — much simpler than the manual `torch.jit.trace` + `coremltools.convert` approach in app-01
- Built NMS into the CoreML model (`nms=True`) — Swift inference code reads final boxes directly, no post-processing needed
- Benchmarked across compute unit configs — `ALL` routes to the Neural Engine automatically and is the fastest option
- Ran a full end-to-end demo inference and parsed the output format

**Key numbers:**
- Model size: ~6MB — suitable for on-device deployment
- YOLOv8n: 3.2M params vs DETR's 167M — 50x smaller

**Why DETR didn't make it to CoreML:**
- Dynamic control flow in the Hungarian matcher breaks `torch.jit.trace`
- Would require `torch.jit.script` and significant model surgery
- Even if exported, 167M params is too large for real-time iPhone inference

**Next step:** iPhone app — load `yolov8n.mlpackage` in Swift, pass `CVPixelBuffer` frames from the camera, draw bounding boxes in real time.

[&#x2191; Back to top](#notebook-04----coreml-export)

---